In [ ]:
# Incident Analysis Notebook
# --------------------------
# Tech stack: SQL (sqlite3) + Python (pandas, matplotlib, seaborn)

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Create database connection and import data
conn = sqlite3.connect(":memory:")  # in-memory database
cursor = conn.cursor()

# Read SQL file (sir_incidents.sql)
with open("sir_incidents.sql", "r", encoding="utf-8") as f:
    sql_script = f.read()
cursor.executescript(sql_script)

# 2. Query data and load into Pandas
df = pd.read_sql_query("SELECT * FROM sir_incidents;", conn)
print(df.head())

# 3. Category and severity statistics
category_counts = df['category'].value_counts()
severity_counts = df['severity'].value_counts()

print("Category distribution:\n", category_counts)
print("Severity distribution:\n", severity_counts)

# 4. Visualization: Category distribution
plt.figure(figsize=(6,4))
sns.barplot(x=category_counts.index, y=category_counts.values, palette="Set2")
plt.title("Incident Count by Category")
plt.ylabel("Count")
plt.xlabel("Category")
plt.show()

# 5. Visualization: Severity distribution
plt.figure(figsize=(6,4))
sns.barplot(x=severity_counts.index, y=severity_counts.values, palette="Set1")
plt.title("Incident Count by Severity")
plt.ylabel("Count")
plt.xlabel("Severity")
plt.show()

# 6. Time trend analysis
df['incident_time'] = pd.to_datetime(df['incident_time'])
df['date'] = df['incident_time'].dt.date

daily_counts = df.groupby(['date','severity']).size().reset_index(name='count')

plt.figure(figsize=(10,5))
sns.lineplot(data=daily_counts, x='date', y='count', hue='severity', marker="o")
plt.title("Daily Incident Trend by Severity")
plt.ylabel("Incident Count")
plt.xlabel("Date")
plt.show()

# 7. Case study: blackout incident (id=32)
blackout = df[df['issue_type']=="system_blackout"]
print("Blackout case:\n", blackout[['incident_time','description']])
